# D1 · 序貫「偷看」資料與最優停止 — 貝葉斯不是萬靈丹

> 業界常宣稱「貝葉斯 A/B 可以隨時偷看資料」。這**常被過度宣傳**。
> 本 notebook 用模擬把真相量出來：偷看如何膨脹假陽性，以及貝葉斯後驗門檻**是否免疫**（答案：否）。

方法論驗證**必須用模擬**——因為只有模擬能讓我們知道真相（計劃書 §5）。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import experiments as ex
from bayes_ab import Prior
rng = np.random.default_rng(42)

## 1 · 模擬 A/A 測試

兩組其實**完全相同**（$p_A=p_B=0.05$），每組每天 500 次曝光、追蹤 28 天、3000 次獨立測試。
既然兩組相同，**任何「宣布勝利」都是假陽性**。

In [2]:
aa = ex.Scenario(pA=0.05, pB=0.05, per_day=500, days=28)
kA, kB = ex.simulate_cumulative(aa, 3000, rng)
table = ex.build_peek_table(kA, kB, aa.per_day, Prior(1, 1))
print('模擬完成，table 形狀 =', table.p_value.shape, '(模擬數 × 天數)')

模擬完成，table 形狀 = (3000, 28) (模擬數 × 天數)


## 2 · 偷看次數 vs. 假陽性率

在**同樣 28 天視野**內，比較不同「偷看次數」下三種宣布規則的假陽性率。

In [3]:
rules = [ex.RuleSpec('freq', 0.05, 'Frequentist p<0.05'),
         ex.RuleSpec('bayes', 0.95, 'Bayes P>0.95'),
         ex.RuleSpec('bayes', 0.975, 'Bayes P>0.975')]
fpr = ex.fpr_vs_looks(table, aa.days, [1, 2, 4, 7, 14, 28], rules)
print(f"{'偷看次數':>8} | " + ' | '.join(f'{r.label:>18}' for r in rules))
for i, K in enumerate(fpr['looks']):
    print(f'{K:>8} | ' + ' | '.join(f"{fpr['rates'][r.label][i]*100:>17.1f}%" for r in rules))

    偷看次數 | Frequentist p<0.05 |       Bayes P>0.95 |      Bayes P>0.975
       1 |               5.0% |               9.5% |               4.7%
       2 |               9.8% |              18.7% |               9.5%
       4 |              15.4% |              27.4% |              15.2%
       7 |              19.5% |              34.0% |              19.2%
      14 |              24.3% |              41.2% |              24.1%
      28 |              28.1% |              46.6% |              27.8%


![偷看膨脹假陽性](../figures/02_peeking_false_positive.png)

**關鍵解讀**（左圖）：
- 只看 1 次（固定視野）：freq p<0.05 ≈ **5%**、Bayes P>0.975 ≈ **5%**（門檻匹配，兩者本就對應）。
- 逐日偷看 28 次：freq ≈ **26%**、Bayes P>0.975 ≈ **26%**——**幾乎一模一樣**。
  → **貝葉斯後驗門檻對偷看一樣不免疫。**
- Bayes P>0.95 ≈ **44%**：門檻較鬆（見下），膨脹更嚴重。

右圖：A/A 下後驗 $P(B>A)$ 每日亂走，約三成的測試曾越過 0.975 / 0.025 門檻。

## 3 · 為什麼？門檻其實是對應的

$P(\theta_B>\theta_A)>0.975$（雙向）在數學上 $\approx$ 雙尾 $\alpha=0.05$；
$P>0.95$（雙向）$\approx$ 雙尾 $\alpha=0.10$。所以「貝葉斯門檻」和「p 值門檻」是同一件事的兩種說法，
**偷看造成的多重比較問題對兩者一視同仁**。這解釋了上圖為何兩條線疊在一起。

## 4 · 那期望損失規則呢？

計劃書要問：同樣偷看下，**貝葉斯期望損失規則**的假陽性率是多少？

In [4]:
eps = 2e-4  # 0.02pp
el = ex.RuleSpec('exploss', eps, 'Expected loss<eps')
el_fixed = ex.declare_rate(ex.fires_for_rule(table, ex.even_schedule(aa.days, 1), el))
el_daily = ex.declare_rate(ex.fires_for_rule(table, np.arange(aa.days), el))
print(f'期望損失規則(ε={eps:.0e})：固定視野={el_fixed:.1%} · 逐日偷看={el_daily:.1%}')

期望損失規則(ε=2e-04)：固定視野=6.6% · 逐日偷看=69.8%


> **但這是另一種『錯』**：期望損失規則只在「後悔量已 < $\varepsilon$」時上線。
> A/A 下兩組本就相同，後悔確實可忽略——所以「上線」其實是決策論意義的**『不在乎誰贏』**，
> 而不是 p 值意義的「錯誤宣稱有差異」。把它和顯著性規則放同一條軸比較會誤導，故上面主圖不含它。
> 它真正的用途是**控制後悔的大小**（見 NB1 第 4 節：何時值得上線）。

## 5 · 誠實的另一面：真有效果時，偷看反而更快

偷看不是一無是處。當 B **真的**好 10% 時，看偷看的偵測速度：

In [5]:
h1 = ex.Scenario(pA=0.05, pB=0.055, per_day=500, days=28)
kA1, kB1 = ex.simulate_cumulative(h1, 3000, rng)
t1 = ex.build_peek_table(kA1, kB1, h1.per_day, Prior(1, 1))
fixed = ex._fires_freq(t1, ex.even_schedule(h1.days, 1), 0.05).any(1).mean()
fires_daily = ex._fires_freq(t1, np.arange(h1.days), 0.05)
daily = fires_daily.any(1).mean()
first = np.where(fires_daily.any(1), fires_daily.argmax(1)+1, h1.days).mean()
print(f'真實提升 +10%：固定視野偵測率={fixed:.1%} · 逐日偷看偵測率={daily:.1%} · 平均第 {first:.1f} 天宣布')

真實提升 +10%：固定視野偵測率=6.4% · 逐日偷看偵測率=66.0% · 平均第 16.9 天宣布


→ 逐日偷看偵測得**快得多**（固定視野在 28 天內常常還來不及達到顯著）。
**取捨很清楚**：偷看換到速度，代價是 A/A 下的假陽性膨脹。

## 6 · 那到底該怎麼辦？

1. **預先固定樣本量 / 視野**（最簡單、最穩）。
2. 若真的需要邊看邊停 → 用 **always-valid inference**：mSPRT、confidence sequences、或有序貫設計的貝葉斯方法。
3. 貝葉斯的真正價值**不是**「免疫偷看」，而是**回答對的問題**（P(B>A)、期望 $ 損失），且在任何一個固定的觀測點都有正確解讀。

## 重點

> 貝葉斯量（後驗機率、期望損失）的**解讀**在偷看下不失效；
> 但把它**門檻化當停止規則**，就重新引入了多重比較 → 長期錯誤率照樣膨脹。
> **能講清楚這個細節，面試會明顯高出一截。**